# Figures for HPML Paper
Optimization of SLMs for Efficient Code Generation

Generates all publication figures and saves them to `outputs/figures/`.

In [ ]:
import json
import os
from collections import defaultdict

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
import pandas as pd

plt.style.use('seaborn-v0_8-whitegrid')

COLORS = {
    'Base SLM':          '#4C72B0',
    'Control SFT':       '#DD8452',
    'Runtime-Aware SFT': '#55A868',
}
OUT_DIR = '../outputs/figures'
os.makedirs(OUT_DIR, exist_ok=True)

def save(name):
    plt.tight_layout()
    plt.savefig(f'{OUT_DIR}/{name}', dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Saved {OUT_DIR}/{name}')

In [ ]:
# Load serving benchmark data
serving = pd.read_csv('../outputs/serving_full_results.csv')

def label_model(path):
    if 'runtime_aware' in path:
        return 'Runtime-Aware SFT'
    if 'control' in path:
        return 'Control SFT'
    return 'Base SLM'

serving['model'] = serving['model_path'].apply(label_model)
serving['backend_label'] = serving['backend'].str.upper()

# Load benchmarked candidates
candidates = []
with open('../outputs/benchmarked_candidates_full.jsonl') as f:
    for line in f:
        candidates.append(json.loads(line))
candidates_df = pd.DataFrame(candidates)
passing = candidates_df[candidates_df['benchmark_passed'] == True].copy()

print(f'Serving rows: {len(serving)}')
print(f'Benchmarked candidates: {len(candidates_df)}, passing: {len(passing)}')

In [ ]:
# Ablation results from clean test split (666 problems, retrained on train/ split)
ablation = {
    'model':               ['Base SLM', 'Control SFT', 'Runtime-Aware SFT'],
    'pass_at_1':           [0.345,       0.642,          0.639],
    'median_exec_time_ms': [0.0819,      0.0807,         0.0809],  # converted to ms
    'avg_latency_s':       [0.787,       0.679,          0.687],
}

In [ ]:
# Figure 1: Pass@1 Comparison
fig, ax = plt.subplots(figsize=(7, 4))

models = ablation['model']
scores = ablation['pass_at_1']
colors = [COLORS[m] for m in models]
bars = ax.bar(models, scores, color=colors, width=0.5, edgecolor='white', linewidth=1.2)

for bar, score in zip(bars, scores):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.01,
            f'{score:.3f}', ha='center', va='bottom', fontsize=11, fontweight='bold')

ax.set_ylim(0, 0.82)
ax.set_ylabel('Pass@1', fontsize=12)
ax.set_title('Code Correctness: Pass@1 by Model', fontsize=13, fontweight='bold')
ax.tick_params(axis='x', labelsize=11)
ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))

save('fig1_pass_at_1.png')

In [ ]:
# Figure 2: Generation Latency Comparison
fig, ax = plt.subplots(figsize=(7, 4))

latencies = ablation['avg_latency_s']
bars = ax.bar(models, latencies, color=colors, width=0.5, edgecolor='white', linewidth=1.2)

for bar, val in zip(bars, latencies):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.008,
            f'{val:.3f}s', ha='center', va='bottom', fontsize=11, fontweight='bold')

ax.set_ylim(0, 1.0)
ax.set_ylabel('Avg Generation Latency (s)', fontsize=12)
ax.set_title('Generation Latency per Problem by Model', fontsize=13, fontweight='bold')
ax.tick_params(axis='x', labelsize=11)

save('fig2_generation_latency.png')

In [ ]:
# Figure 3: Serving Throughput — HF vs vLLM
fig, ax = plt.subplots(figsize=(9, 4))

serving_models = ['Base SLM', 'Control SFT', 'Runtime-Aware SFT']
x = np.arange(len(serving_models))
width = 0.35

hf_vals   = [serving[(serving['model'] == m) & (serving['backend'] == 'hf')]['throughput_output_tokens_per_s'].values[0] for m in serving_models]
vllm_vals = [serving[(serving['model'] == m) & (serving['backend'] == 'vllm')]['throughput_output_tokens_per_s'].values[0] for m in serving_models]

bars_hf   = ax.bar(x - width/2, hf_vals,   width, label='HuggingFace', color='#4878CF', edgecolor='white')
bars_vllm = ax.bar(x + width/2, vllm_vals, width, label='vLLM',        color='#6ACC65', edgecolor='white')

for bar, val in zip(bars_hf, hf_vals):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 20,
            f'{val:.0f}', ha='center', va='bottom', fontsize=9)
for bar, val in zip(bars_vllm, vllm_vals):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 20,
            f'{val:.0f}', ha='center', va='bottom', fontsize=9)

ax.set_xticks(x)
ax.set_xticklabels(serving_models, fontsize=11)
ax.set_ylabel('Throughput (tokens/s)', fontsize=12)
ax.set_title('Serving Throughput: HuggingFace vs vLLM', fontsize=13, fontweight='bold')
ax.legend(fontsize=11)

save('fig3_serving_throughput.png')

In [ ]:
# Figure 4: GPU Utilization — HF vs vLLM
fig, ax = plt.subplots(figsize=(9, 4))

hf_util   = [serving[(serving['model'] == m) & (serving['backend'] == 'hf')]['avg_gpu_util_pct'].values[0] for m in serving_models]
vllm_util = [serving[(serving['model'] == m) & (serving['backend'] == 'vllm')]['avg_gpu_util_pct'].values[0] for m in serving_models]

bars_hf   = ax.bar(x - width/2, hf_util,   width, label='HuggingFace', color='#4878CF', edgecolor='white')
bars_vllm = ax.bar(x + width/2, vllm_util, width, label='vLLM',        color='#6ACC65', edgecolor='white')

for bar, val in zip(bars_hf, hf_util):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            f'{val:.1f}%', ha='center', va='bottom', fontsize=9)
for bar, val in zip(bars_vllm, vllm_util):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            f'{val:.1f}%', ha='center', va='bottom', fontsize=9)

ax.set_xticks(x)
ax.set_xticklabels(serving_models, fontsize=11)
ax.set_ylim(0, 115)
ax.set_ylabel('Avg GPU Utilization (%)', fontsize=12)
ax.set_title('GPU Utilization: HuggingFace vs vLLM', fontsize=13, fontweight='bold')
ax.legend(fontsize=11)

save('fig4_gpu_utilization.png')

In [ ]:
# Figure 5: Execution Time Distribution (log scale, trimmed to data range)
fig, ax = plt.subplots(figsize=(8, 4))

times_ms = passing['median_time'].values * 1000  # convert to ms

# Trim to 99th percentile to avoid empty tail
p99 = np.percentile(times_ms, 99)
trimmed = times_ms[times_ms <= p99]
n_clipped = len(times_ms) - len(trimmed)

log_bins = np.logspace(np.log10(trimmed.min()), np.log10(trimmed.max()), 45)
ax.hist(trimmed, bins=log_bins, color='#4C72B0', edgecolor='white', alpha=0.85)

ax.set_xscale('log')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:.3g}'))
ax.set_xlabel('Median Execution Time (ms, log scale)', fontsize=12)
ax.set_ylabel('Number of Candidates', fontsize=12)
ax.set_title(f'Distribution of Solution Execution Times\n(7,992 passing candidates, {n_clipped} outliers >p99 excluded)', fontsize=13, fontweight='bold')

ax.axvline(np.median(times_ms), color='#DD8452', linestyle='--', linewidth=1.8,
           label=f'Median: {np.median(times_ms):.3f} ms')
ax.legend(fontsize=11)

save('fig5_execution_time_distribution.png')

In [ ]:
# Figure 6: Speedup Distribution (fastest vs first-correct candidate)
by_problem = defaultdict(list)
for _, row in passing.iterrows():
    by_problem[row['dataset_index']].append(row)

speedups = []
for idx, rows in by_problem.items():
    rows_sorted_by_id   = sorted(rows, key=lambda r: r['candidate_id'])
    rows_sorted_by_time = sorted(rows, key=lambda r: r['median_time'])
    first_time   = rows_sorted_by_id[0]['median_time']
    fastest_time = rows_sorted_by_time[0]['median_time']
    if fastest_time and fastest_time > 0:
        speedups.append(first_time / fastest_time)

speedups = np.array(speedups)
print(f'Problems: {len(speedups)}')
print(f'Median speedup: {np.median(speedups):.3f}x')
print(f'Mean speedup:   {np.mean(speedups):.3f}x')
print(f'Problems >1.5x: {(speedups >= 1.5).sum()} ({(speedups >= 1.5).mean()*100:.1f}%)')

# Clip x-axis at 2.0 to zoom into where the data actually is
clip_at = 2.0
n_beyond = (speedups > clip_at).sum()
fig, ax = plt.subplots(figsize=(8, 4))

bins = np.linspace(1.0, clip_at, 40)
ax.hist(np.clip(speedups, 1.0, clip_at), bins=bins, color='#4C72B0', edgecolor='white', alpha=0.85)

ax.axvline(np.median(speedups), color='#55A868', linestyle='--', linewidth=1.8,
           label=f'Median: {np.median(speedups):.2f}x')
ax.text(0.97, 0.92, f'{n_beyond} problems >{clip_at}x\n(not shown)',
        transform=ax.transAxes, ha='right', va='top', fontsize=9, color='gray')

ax.set_xlim(1.0, clip_at)
ax.set_xlabel('Speedup (first-correct time / fastest-correct time)', fontsize=12)
ax.set_ylabel('Number of Problems', fontsize=12)
ax.set_title('Runtime-Aware Training Signal: Speedup Distribution\n(nearly all problems cluster near 1.0x)', fontsize=13, fontweight='bold')
ax.legend(fontsize=11)

save('fig6_speedup_distribution.png')

In [ ]:
# Figure 9: Memory vs Throughput Tradeoff (HF vs vLLM)
fig, ax = plt.subplots(figsize=(8, 5))

backend_colors = {'hf': '#4878CF', 'vllm': '#6ACC65'}
markers = {'Base SLM': 'o', 'Control SFT': 's', 'Runtime-Aware SFT': '^'}
marker_size = 140

# Custom label offsets (points) to avoid overlap in tight clusters
# HF points are ~same memory, slightly different throughput → stagger vertically
# vLLM points are ~same memory, different throughput → stagger horizontally
label_offsets = {
    ('Base SLM',          'hf'):   (-10, -28),
    ('Runtime-Aware SFT', 'hf'):   (-10,  12),
    ('Base SLM',          'vllm'): (-90, -28),
    ('Runtime-Aware SFT', 'vllm'): (-90,  12),
    ('Control SFT',       'hf'):   (-10,  -8),
    ('Control SFT',       'vllm'): (-90,  -8),
}

for _, row in serving.iterrows():
    x_val = row['peak_cuda_memory_mb'] / 1024
    y_val = row['throughput_output_tokens_per_s']
    key = (row['model'], row['backend'])
    dx, dy = label_offsets.get(key, (8, 4))

    ax.scatter(x_val, y_val,
               color=backend_colors[row['backend']],
               marker=markers[row['model']],
               s=marker_size, zorder=5,
               edgecolors='white', linewidths=0.8)
    ax.annotate(
        row['model'],
        (x_val, y_val),
        xytext=(dx, dy),
        textcoords='offset points',
        fontsize=8.5,
        arrowprops=dict(arrowstyle='-', color='gray', lw=0.6),
    )

# Cluster labels
hf_mem   = serving[serving['backend'] == 'hf']['peak_cuda_memory_mb'].mean() / 1024
hf_thr   = serving[serving['backend'] == 'hf']['throughput_output_tokens_per_s'].mean()
vllm_mem = serving[serving['backend'] == 'vllm']['peak_cuda_memory_mb'].mean() / 1024
vllm_thr = serving[serving['backend'] == 'vllm']['throughput_output_tokens_per_s'].mean()

ax.text(hf_mem, hf_thr + 70, 'HuggingFace\n(low memory,\nlow throughput)',
        ha='center', fontsize=9, color='#4878CF', fontweight='bold')
ax.text(vllm_mem, vllm_thr - 200, 'vLLM\n(high memory,\nhigh throughput)',
        ha='center', fontsize=9, color='#3a9a30', fontweight='bold')

from matplotlib.lines import Line2D
legend_elements = [
    Line2D([0], [0], marker='o', color='w', markerfacecolor='#4878CF', markersize=9, label='HuggingFace'),
    Line2D([0], [0], marker='o', color='w', markerfacecolor='#6ACC65', markersize=9, label='vLLM'),
    Line2D([0], [0], marker='o', color='gray', markersize=7, linestyle='None', label='Base SLM'),
    Line2D([0], [0], marker='s', color='gray', markersize=7, linestyle='None', label='Control SFT'),
    Line2D([0], [0], marker='^', color='gray', markersize=7, linestyle='None', label='Runtime-Aware SFT'),
]
ax.legend(handles=legend_elements, fontsize=9, loc='upper left')

ax.set_xlabel('Peak GPU Memory (GB)', fontsize=12)
ax.set_ylabel('Throughput (tokens/s)', fontsize=12)
ax.set_title('Memory–Throughput Tradeoff: HuggingFace vs vLLM', fontsize=13, fontweight='bold')

save('fig9_memory_throughput_tradeoff.png')

In [ ]:
# Figure 8: Combined Ablation — Pass@1 + Generation Latency side by side
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

# Left: Pass@1
bars = ax1.bar(models, ablation['pass_at_1'], color=colors, width=0.5, edgecolor='white', linewidth=1.2)
for bar, val in zip(bars, ablation['pass_at_1']):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
             f'{val:.1%}', ha='center', va='bottom', fontsize=11, fontweight='bold')
ax1.set_ylim(0, 0.82)
ax1.set_ylabel('Pass@1', fontsize=12)
ax1.set_title('(a) Code Correctness', fontsize=12, fontweight='bold')
ax1.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))
ax1.tick_params(axis='x', labelsize=10)

# Right: Generation latency
bars2 = ax2.bar(models, ablation['avg_latency_s'], color=colors, width=0.5, edgecolor='white', linewidth=1.2)
for bar, val in zip(bars2, ablation['avg_latency_s']):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.008,
             f'{val:.3f}s', ha='center', va='bottom', fontsize=11, fontweight='bold')
ax2.set_ylim(0, 1.0)
ax2.set_ylabel('Avg Generation Latency (s)', fontsize=12)
ax2.set_title('(b) Generation Latency', fontsize=12, fontweight='bold')
ax2.tick_params(axis='x', labelsize=10)

fig.suptitle('Effect of Fine-Tuning on Correctness and Latency', fontsize=13, fontweight='bold', y=1.02)

save('fig8_combined_ablation.png')

In [ ]:
# Figure 7: Per-Prompt Latency — HF vs vLLM (21x speedup headline)
fig, ax = plt.subplots(figsize=(9, 4))

hf_lat   = [serving[(serving['model'] == m) & (serving['backend'] == 'hf')]['avg_latency_per_prompt_s'].values[0] for m in serving_models]
vllm_lat = [serving[(serving['model'] == m) & (serving['backend'] == 'vllm')]['avg_latency_per_prompt_s'].values[0] for m in serving_models]

bars_hf   = ax.bar(x - width/2, hf_lat,   width, label='HuggingFace', color='#4878CF', edgecolor='white')
bars_vllm = ax.bar(x + width/2, vllm_lat, width, label='vLLM',        color='#6ACC65', edgecolor='white')

for bar, val in zip(bars_hf, hf_lat):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
            f'{val:.3f}s', ha='center', va='bottom', fontsize=9)
for i, (bar, val) in enumerate(zip(bars_vllm, vllm_lat)):
    speedup = hf_lat[i] / val
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
            f'{val:.3f}s\n({speedup:.0f}x faster)', ha='center', va='bottom', fontsize=8, color='#2a7a2a')

ax.set_xticks(x)
ax.set_xticklabels(serving_models, fontsize=11)
ax.set_ylabel('Avg Latency per Prompt (s)', fontsize=12)
ax.set_title('Per-Prompt Serving Latency: HuggingFace vs vLLM', fontsize=13, fontweight='bold')
ax.legend(fontsize=11)

save('fig7_serving_latency.png')